# Finetuning Gemma

## Installing the Libraries

In [ ]:
!pip install unsloth trl

In [ ]:
!pip install -U "transformers==5.14.1"

## Importing Hugging Face Access Token

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
WANDB_API_KEY = userdata.get('WANDB_API_KEY')

## Logging in to Hugging Face

In [ ]:
from huggingface_hub import login

In [ ]:
login(HF_TOKEN)

## Logging in to WANDB

In [ ]:
import os
import wandb

In [ ]:
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
wandb.login()

## Importing the Libraries

In [ ]:
import json
import torch
from datetime import datetime
from datasets import Dataset
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig

## Importing the dataset

In [ ]:
FILENAME = "Finetune Dataset.jsonl"
TRAIN_EXAMPLES = 900

with open(FILENAME, "r", encoding = "utf-8") as f:
  lines = f.readlines()
  train_dataset = lines[:TRAIN_EXAMPLES]
  valid_dataset = lines[TRAIN_EXAMPLES:]

train_dataset = [json.loads(x) for x in train_dataset]
valid_dataset = [json.loads(x) for x in valid_dataset]

train_dataset = Dataset.from_list([{"text": f"###User:\n{x['prompt']}\n\n###Assistant:\n{x['completion']}"} for x in train_dataset])
valid_dataset = Dataset.from_list([{"text": f"###User:\n{x['prompt']}\n\n###Assistant:\n{x['completion']}"} for x in valid_dataset])

## Configurations

#### To Calculate Warm-up Steps:
```python
steps_per_epoch = math.ceil(TRAIN_EXAMPLES / BATCH_SIZE / GRADIENT_ACCUMULATION_STEPS)
total_training_steps = steps_per_epoch * EPOCHS
WARMUP_STEPS = max(1, int(total_training_steps * 0.01))

In [ ]:
BASE_MODEL = "unsloth/gemma-4-12B-it"
PROJECT_PREFIX = "Gemma-Finetune"
HF_USERNAME = "username"
WANDB_ENTITY = "usernameteam"

TIMESTAMP = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_NAME = f"{PROJECT_PREFIX}-{TIMESTAMP}"
HUB_MODEL_NAME = f"{HF_USERNAME}/{PROJECT_NAME}"

EPOCHS = 3
BATCH_SIZE = 16
GRADIENT_ACCUMULATION = 2
MAX_SEQUENCE_LENGTH = 288
SAVE_STEPS = 10

# QLoRA Config

LORA_R = 32
LORA_ALPHA = LORA_R * 2
LORA_DROPOUT = 0.2

ATTENTION_LAYERS = ["q_proj", "k_proj", "v_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]

TARGET_MODULES = ATTENTION_LAYERS + MLP_LAYERS

# Hyper-parameters

LEARNING_RATE = 1e-4
WARMUP_STEPS = 1
LR_SCHEDULER_TYPE = "cosine"
OPTIMIZER = "adamw_torch"
WEIGHT_DECAY = 0.001
MAX_GRAD_NORM = 0.3

## Starting Weight & Biases Session

In [ ]:
os.environ["WANDB_PROJECT"] = PROJECT_PREFIX
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

In [ ]:
wandb.init(
    project = PROJECT_PREFIX,
    name = PROJECT_NAME,
    entity = WANDB_ENTITY,
)

## Loading Quantized Model from Unsloth

In [ ]:
base_model, tokenizer = FastModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQUENCE_LENGTH,
    dtype = None,
    load_in_4bit = True
)

In [ ]:
print(base_model)

Gemma4UnifiedForConditionalGeneration(
  (model): Gemma4UnifiedModel(
    (language_model): Gemma4UnifiedTextModel(
      (embed_tokens): Gemma4UnifiedTextScaledWordEmbedding(262144, 3840, padding_idx=0)
      (layers): ModuleList(
        (0-4): 5 x Gemma4UnifiedTextDecoderLayer(
          (self_attn): Gemma4UnifiedTextAttention(
            (q_proj): Linear4bit(in_features=3840, out_features=4096, bias=False)
            (q_norm): Gemma4UnifiedRMSNorm()
            (k_norm): Gemma4UnifiedRMSNorm()
            (v_norm): Gemma4UnifiedRMSNorm()
            (k_proj): Linear4bit(in_features=3840, out_features=2048, bias=False)
            (v_proj): Linear4bit(in_features=3840, out_features=2048, bias=False)
            (o_proj): Linear4bit(in_features=4096, out_features=3840, bias=False)
          )
          (mlp): Gemma4UnifiedTextMLP(
            (gate_proj): Linear4bit(in_features=3840, out_features=15360, bias=False)
            (up_proj): Linear4bit(in_features=3840, out_features=15

## Configuring LoRA Parameters

In [ ]:
base_model = FastModel.get_peft_model(
    base_model,
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    target_modules = TARGET_MODULES,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

## Configuring Training Parameters

In [ ]:
training_parameters = SFTConfig(
    output_dir = PROJECT_NAME,
    num_train_epochs = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,

    learning_rate = LEARNING_RATE,
    optim = OPTIMIZER,
    weight_decay = WEIGHT_DECAY,
    max_grad_norm = MAX_GRAD_NORM,
    gradient_accumulation_steps = GRADIENT_ACCUMULATION,

    warmup_steps = WARMUP_STEPS,
    lr_scheduler_type = LR_SCHEDULER_TYPE,

    report_to = "wandb",
    run_name = PROJECT_NAME,
    max_length = MAX_SEQUENCE_LENGTH,

    save_strategy = "steps",
    save_steps = SAVE_STEPS,
    save_total_limit = 10,

    eval_strategy = "steps",
    eval_steps = SAVE_STEPS,

    push_to_hub = True,
    hub_model_id = HUB_MODEL_NAME,
    hub_private_repo = True
)

## Creating the Trainer

In [ ]:
finetuning = SFTTrainer(
    model = base_model,
    tokenizer = tokenizer,
    dataset_text_field = "text",
    train_dataset = train_dataset,
    eval_dataset = valid_dataset,
    args = training_parameters,
    packing = False
)

In [ ]:
finetuning.train()

finetuning.model.push_to_hub(PROJECT_NAME, private = True)
print(f"Saved model to hub: {PROJECT_NAME}")

## Statistics

#### Train Summary:

<sub>████████████████████████ 100% | 87/87 | 1:16:27 | Epoch 3/3</sub>

<br>

| Step | Training Loss | Validation Loss |
| ---: | ------------: | --------------: |
|   10 |      1.814657 |        1.778607 |
|   20 |      0.761403 |        0.763173 |
|   30 |      0.634181 |        0.635307 |
|   40 |      0.646047 |        0.715160 |
|   50 |      0.579972 |        0.632861 |
|   60 |      0.541776 |        0.585348 |
|   70 |      0.507801 |        0.600252 |
|   80 |      0.525180 |        0.597759 |
|   87 |      0.644238 |        0.597697 |

<br>

#### Run Summary:

| Metric | Value |
|---|---:|
| `eval/loss` | 0.5977 |
| `eval/runtime` | 52.2095 |
| `eval/samples_per_second` | 1.896 |
| `eval/steps_per_second` | 0.479 |
| `total_flos` | 5.169784629598618e+16 |
| `train/epoch` | 3 |
| `train/global_step` | 87 |
| `train/grad_norm` | 2.05351 |
| `train/learning_rate` | 0.0 |
| `train/loss` | 0.64424 |

## Finishing WANDB Session

In [ ]:
wandb.finish()

## Testing the Fine-Tuned Model

In [ ]:
FastModel.for_inference(base_model)

In [ ]:
prompt = "Extract the relevant fields from this OCR text, ignoring any unrelated text or artifacts:\n\"CITY HEALTH CLINIC\\nBILL NO: HC-48291\\nDATE: 08/MAR/2026\\nPATIENT: SARAH LIM\\nCONSULTATION $55.00\\nMEDICATION $25.00\\nGST (9%): $7.20\\nTOTAL: $87.20\\nREMARKS: FOLLOW-UP VISIT SUBSIDY APPLIED.\\nPAGE 1 OF 1\"\n"

In [ ]:
inputs = tokenizer(
    text = prompt,
    return_tensors = "pt"
).to(base_model.device)

In [ ]:
outputs = base_model.generate(
    **inputs,
    max_new_tokens = 200
)

In [ ]:
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens = True
)

In [ ]:
print(response[response.find("{"):])

{
  "doc_type": "medical_bill",
  "schema": {
    "Date_of_Visit": "08/MAR/2026",
    "Date_of_Visit_confidence_score": 98,
    "Amt": "87.20",
    "Amt_confidence_score": 99,
    "GST": "7.20",
    "GST_confidence_score": 94,
    "Remarks": "FOLLOW-UP VISIT SUBSIDY APPLIED.",
    "Remarks_confidence_score": 89
  }
}
